# ML Assignment 1 — Bike Sharing Demand Prediction

**Course:** Machine Learning (M.Tech AIML/DSE)  
**Objective:** Predict hourly bike rentals using weather, time, and seasonal data.  
**Evaluation Metric:** RMSLE (Root Mean Squared Logarithmic Error)

## 0. Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

print("All imports successful!")

All imports successful!


## 1. Load Data

In [2]:
train = pd.read_csv('data/bike_train.csv')
test = pd.read_csv('data/bike_test.csv')
sample_sub = pd.read_csv('data/sampleSubmission.csv')

print(f"Training set shape: {train.shape}")
print(f"Test set shape:     {test.shape}")
print(f"Sample submission:  {sample_sub.shape}")

Training set shape: (10450, 12)
Test set shape:     (2613, 9)
Sample submission:  (2613, 2)


---
## Exploratory Data Analysis (EDA)

### Q1. Examine dataset size, missing values, and feature types

In [3]:
print(f"Training: {train.shape[0]} rows x {train.shape[1]} cols, Test: {test.shape[0]} rows x {test.shape[1]} cols")
print(f"No missing values in either set.")

Training: 10450 rows x 12 cols, Test: 2613 rows x 9 cols
No missing values in either set.


**Q1 Answer:**

- **Dataset size:** Training set has 10,450 rows and 12 columns. Test set has 2,613 rows and 9 columns.
- **Missing values:** None in either dataset.
- **Feature types:** Categorical (season, holiday, workingday, weather), Continuous (temp, atemp, humidity, windspeed), Datetime, Target (count).

### Q2. Visualize relationships between key features and the target variable (`count`)

In [7]:
train['datetime'] = pd.to_datetime(train['datetime'])
train['hour'] = train['datetime'].dt.hour
train['dayofweek'] = train['datetime'].dt.dayofweek
train['month'] = train['datetime'].dt.month
train['year'] = train['datetime'].dt.year

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.boxplot(x='hour', y='count', data=train, ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Bike Rentals by Hour of Day', fontsize=13)
sns.boxplot(x='season', y='count', data=train, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Bike Rentals by Season', fontsize=13)
sns.boxplot(x='weather', y='count', data=train, ax=axes[1, 0], palette='Set3')
axes[1, 0].set_title('Bike Rentals by Weather', fontsize=13)
sns.boxplot(x='workingday', y='count', data=train, ax=axes[1, 1], palette='pastel')
axes[1, 1].set_title('Working Day vs Non-Working Day', fontsize=13)
plt.tight_layout(); plt.savefig('eda_boxplots.png', dpi=150, bbox_inches='tight'); plt.show()

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(train['temp'], train['count'], alpha=0.15, s=10, c='steelblue')
axes[0].set_xlabel('Temperature'); axes[0].set_ylabel('Count'); axes[0].set_title('Temperature vs Bike Rentals')
axes[1].scatter(train['humidity'], train['count'], alpha=0.15, s=10, c='coral')
axes[1].set_xlabel('Humidity'); axes[1].set_ylabel('Count'); axes[1].set_title('Humidity vs Bike Rentals')
axes[2].scatter(train['windspeed'], train['count'], alpha=0.15, s=10, c='forestgreen')
axes[2].set_xlabel('Windspeed'); axes[2].set_ylabel('Count'); axes[2].set_title('Windspeed vs Bike Rentals')
plt.tight_layout(); plt.savefig('eda_scatter.png', dpi=150, bbox_inches='tight'); plt.show()

In [9]:
hourly_avg = train.groupby(['hour', 'workingday'])['count'].mean().reset_index()
fig, ax = plt.subplots(figsize=(14, 6))
for wd, label, color in [(0, 'Non-Working Day', 'coral'), (1, 'Working Day', 'steelblue')]:
    subset = hourly_avg[hourly_avg['workingday'] == wd]
    ax.plot(subset['hour'], subset['count'], marker='o', label=label, linewidth=2, color=color)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Average Bike Rentals')
ax.set_title('Average Hourly Bike Rentals: Working Day vs Non-Working Day')
ax.legend(); ax.set_xticks(range(24))
plt.tight_layout(); plt.savefig('eda_hourly_pattern.png', dpi=150, bbox_inches='tight'); plt.show()

In [10]:
numerical_cols = ['temp', 'atemp', 'humidity', 'windspeed', 'hour', 'count']
corr_matrix = train[numerical_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('Correlation Matrix')
plt.tight_layout(); plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight'); plt.show()

**Q2 Observations:** Hour of day is the strongest driver. Working days show bimodal peaks at 8AM and 5-6PM. Fall has highest rentals; spring lowest. Clear weather has highest rentals. Temperature is positively correlated; humidity negatively.

### Q3. Which variables are likely to be most informative?

**Q3 Answer:** Ranked by predictive power: 1) Hour of day, 2) Temperature, 3) Season, 4) Working day, 5) Weather, 6) Humidity, 7) Year, 8) Windspeed. Interaction effects (hour x workingday) are also highly informative.

---
## Feature Engineering

### Q4. Derived features and transformations

In [11]:
def engineer_features(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=True, format='mixed')
    df['hour'] = df['datetime'].dt.hour
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['is_rush_morning'] = ((df['hour'] >= 7) & (df['hour'] <= 9) & (df['workingday'] == 1)).astype(int)
    df['is_rush_evening'] = ((df['hour'] >= 16) & (df['hour'] <= 19) & (df['workingday'] == 1)).astype(int)
    df['is_rush_hour'] = (df['is_rush_morning'] | df['is_rush_evening']).astype(int)
    df['temp_humidity'] = df['temp'] * df['humidity']
    df['temp_windspeed'] = df['temp'] * df['windspeed']
    df['temp_sq'] = df['temp'] ** 2
    season_dummies = pd.get_dummies(df['season'], prefix='season', drop_first=True, dtype=int)
    weather_dummies = pd.get_dummies(df['weather'], prefix='weather', drop_first=True, dtype=int)
    hour_dummies = pd.get_dummies(df['hour'], prefix='hour', drop_first=True, dtype=int)
    df = pd.concat([df, season_dummies, weather_dummies, hour_dummies], axis=1)
    return df

train_fe = engineer_features(train)
test_fe = engineer_features(test)
print(f"Original training features: {train.shape[1]}")
print(f"Engineered training features: {train_fe.shape[1]}")

Original training features: 16
Engineered training features: 56


**Q4 Answer:** Applied datetime decomposition, weekend flag, cyclical encoding (sin/cos for hour and month), rush hour indicators, interaction terms (temp*humidity, temp*windspeed), polynomial term (temp^2), and one-hot encoding for season, weather, and hour.

---
## Regression Models

### Q5. Simple Linear Regression

In [12]:
feature_cols_simple = ['temp', 'atemp', 'humidity', 'windspeed', 'holiday', 'workingday']
X_simple = train_fe[feature_cols_simple].copy()
y = train_fe['count'].copy()
X_train_s, X_val_s, y_train, y_val = train_test_split(X_simple, y, test_size=0.2, random_state=42)
lr_simple = LinearRegression()
lr_simple.fit(X_train_s, y_train)
y_pred_train_s = lr_simple.predict(X_train_s)
y_pred_val_s = lr_simple.predict(X_val_s)
print(f"MODEL 1: Simple Linear Regression (raw features only)")
print(f"Training RMSLE:   {rmsle(y_train, y_pred_train_s):.4f}")
print(f"Validation RMSLE: {rmsle(y_val, y_pred_val_s):.4f}")

MODEL 1: Simple Linear Regression (raw features only)
Training RMSLE:   1.4319
Validation RMSLE: 1.3775


**Q5 Answer:** Simple LR with raw features gives RMSLE ~1.38. Too high — cannot capture hour-of-day or non-linear effects.

### Q6. Extended Models

In [13]:
feature_cols_eng = [c for c in train_fe.columns if c not in ['datetime', 'casual', 'registered', 'count']]
X_eng = train_fe[feature_cols_eng].copy()
X_train_e, X_val_e, y_train, y_val = train_test_split(X_eng, y, test_size=0.2, random_state=42)
lr_eng = LinearRegression()
lr_eng.fit(X_train_e, y_train)
y_pred_train_e = lr_eng.predict(X_train_e)
y_pred_val_e = lr_eng.predict(X_val_e)
print(f"MODEL 2: LR + Engineered Features")
print(f"Training RMSLE: {rmsle(y_train, y_pred_train_e):.4f}, Validation RMSLE: {rmsle(y_val, y_pred_val_e):.4f}")

MODEL 2: LR + Engineered Features
Training RMSLE: 1.1391, Validation RMSLE: 1.1296


In [14]:
feature_cols_poly = ['temp', 'atemp', 'humidity', 'windspeed', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'holiday', 'workingday', 'is_weekend', 'is_rush_morning', 'is_rush_evening', 'year']
X_poly_base = train_fe[feature_cols_poly].copy()
X_train_pb, X_val_pb, y_train, y_val = train_test_split(X_poly_base, y, test_size=0.2, random_state=42)
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_train_p2 = poly2.fit_transform(X_train_pb)
X_val_p2 = poly2.transform(X_val_pb)
scaler_p2 = StandardScaler()
X_train_p2_s = scaler_p2.fit_transform(X_train_p2)
X_val_p2_s = scaler_p2.transform(X_val_p2)
lr_poly2 = LinearRegression()
lr_poly2.fit(X_train_p2_s, y_train)
y_pred_train_p2 = lr_poly2.predict(X_train_p2_s)
y_pred_val_p2 = lr_poly2.predict(X_val_p2_s)
print(f"MODEL 3: Polynomial Regression (Degree 2)")
print(f"Training RMSLE: {rmsle(y_train, y_pred_train_p2):.4f}, Validation RMSLE: {rmsle(y_val, y_pred_val_p2):.4f}")

MODEL 3: Polynomial Regression (Degree 2)
Training RMSLE: 1.0337, Validation RMSLE: 0.9850


In [15]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 5000, 10000]
best_ridge_score = float('inf')
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_p2_s, y_train)
    val_rmsle = rmsle(y_val, ridge.predict(X_val_p2_s))
    if val_rmsle < best_ridge_score:
        best_ridge_score = val_rmsle
        best_ridge_alpha = alpha
        best_ridge_model = ridge
print(f"MODEL 4: Ridge (poly, alpha={best_ridge_alpha}) \u2014 Best Val RMSLE: {best_ridge_score:.4f}")

MODEL 4: Ridge (poly, alpha=10000) — Best Val RMSLE: 0.9487


In [16]:
plt.figure(figsize=(10, 5))
plt.title('Ridge Regression: RMSLE vs Regularization Strength')
plt.tight_layout(); plt.savefig('ridge_tuning.png', dpi=150, bbox_inches='tight'); plt.show()

In [17]:
lasso_alphas = [0.001, 0.01, 0.1, 1, 5, 10, 50, 100]
best_lasso_score = float('inf')
for alpha in lasso_alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_p2_s, y_train)
    val_rmsle_l = rmsle(y_val, lasso.predict(X_val_p2_s))
    if val_rmsle_l < best_lasso_score:
        best_lasso_score = val_rmsle_l
        best_lasso_alpha = alpha
        best_lasso_model = lasso
print(f"MODEL 5: Lasso (poly, alpha={best_lasso_alpha}) \u2014 Best Val RMSLE: {best_lasso_score:.4f}, Non-zero: {np.sum(best_lasso_model.coef_ != 0)}/{len(best_lasso_model.coef_)}")

MODEL 5: Lasso (poly, alpha=10) — Best Val RMSLE: 0.9916, Non-zero: 22/119


In [18]:
y_log = np.log1p(y)
X_train_el, X_val_el, y_train_log, y_val_log = train_test_split(X_eng, y_log, test_size=0.2, random_state=42)
_, _, y_train_orig, y_val_orig = train_test_split(X_eng, y, test_size=0.2, random_state=42)
lr_log = LinearRegression()
lr_log.fit(X_train_el, y_train_log)
y_pred_train_log = np.expm1(lr_log.predict(X_train_el))
y_pred_val_log = np.expm1(lr_log.predict(X_val_el))
print(f"MODEL 6: LR + Log Target + Engineered Features")
print(f"Training RMSLE: {rmsle(y_train_orig, y_pred_train_log):.4f}, Validation RMSLE: {rmsle(y_val_orig, y_pred_val_log):.4f}")

MODEL 6: LR + Log Target + Engineered Features
Training RMSLE: 0.5664, Validation RMSLE: 0.5485


In [19]:
scaler_eng = StandardScaler()
X_train_es = scaler_eng.fit_transform(X_train_el)
X_val_es = scaler_eng.transform(X_val_el)
alphas_final = [0.01, 0.1, 1, 5, 10, 50, 100, 500, 1000]
best_final_score = float('inf')
for alpha in alphas_final:
    ridge_f = Ridge(alpha=alpha)
    ridge_f.fit(X_train_es, y_train_log)
    val_rmsle_f = rmsle(y_val_orig, np.expm1(ridge_f.predict(X_val_es)))
    if val_rmsle_f < best_final_score:
        best_final_score = val_rmsle_f
        best_final_alpha = alpha
        best_final_model = ridge_f
print(f"MODEL 7: Ridge + Log + Eng (alpha={best_final_alpha}) \u2014 Best Val RMSLE: {best_final_score:.4f} \u2605")

MODEL 7: Ridge + Log + Eng (alpha=0.01) — Best Val RMSLE: 0.5485 ★


---
## Model Comparison

### Q7. Summary

| Model | Train RMSLE | Val RMSLE |
|-------|------------|-----------|
| Simple LR | 1.4319 | 1.3775 |
| LR + Engineered | 1.1391 | 1.1296 |
| Polynomial (deg 2) | 1.0337 | 0.9850 |
| Ridge (poly) | 0.9720 | 0.9487 |
| Lasso (poly) | 1.0146 | 0.9916 |
| LR + Log Target | 0.5664 | 0.5485 |
| Ridge + Log + Eng | 0.5664 | 0.5485 ★ |

In [21]:
fig, ax = plt.subplots(figsize=(12, 6))
models = ['M1:Simple LR', 'M2:LR+Eng', 'M3:Poly2', 'M4:Ridge+Poly', 'M5:Lasso+Poly', 'M6:LR+Log', 'M7:Best']
val_scores = [1.3775, 1.1296, 0.9850, 0.9487, 0.9916, 0.5485, 0.5485]
ax.bar(range(len(models)), val_scores, color='coral', alpha=0.8)
ax.set_xlabel('Model'); ax.set_ylabel('Val RMSLE'); ax.set_title('Model Comparison')
ax.set_xticks(range(len(models))); ax.set_xticklabels(models, rotation=30)
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

### Q8. Residual Plot for Best Model

In [22]:
y_pred_val_best = np.expm1(best_final_model.predict(X_val_es))
residuals = y_val_orig.values - y_pred_val_best
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(y_pred_val_best, residuals, alpha=0.3, s=15, c='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=50, edgecolor='black', color='steelblue', alpha=0.7)
axes[1].set_title('Residual Distribution')
from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot')
plt.tight_layout(); plt.savefig('residual_plots.png', dpi=150, bbox_inches='tight'); plt.show()
print(f"Residual Stats: Mean={np.mean(residuals):.2f}, Std={np.std(residuals):.2f}, Min={np.min(residuals):.2f}, Max={np.max(residuals):.2f}")

Residual Stats: Mean=11.38, Std=70.45, Min=-470.09, Max=431.29


**Q8:** Residuals show some heteroscedasticity (variance increases with predicted values). Distribution is approximately normal with slight right skew. Q-Q plot confirms non-normality at tails.

### Q9. Why does the winning model perform better?

The best model (Ridge + Log Target + Engineered Features) wins because: 1) Log transform aligns with RMSLE metric and handles skewness, 2) Engineered features capture hour-of-day effects via dummies, rush hour interactions, cyclical encoding, and non-linear temperature effects, 3) Ridge regularization prevents overfitting, 4) Feature scaling ensures proportional contribution.

---
## Reflection Questions

### Q10. RMSLE vs RMSE

RMSLE operates on log scale, penalizing under-predictions more than over-predictions of the same magnitude. It emphasizes percentage-like errors rather than absolute errors, appropriate for count data with varying scale.

### Q11. Simplicity vs Power

Simple models have low variance but high bias. Complex models capture non-linear patterns but risk overfitting. Regularization (Ridge, Lasso) provides a principled trade-off.

### Q12. Why Linear Regression Fails for Time-of-Day

LR assumes monotonic relationships. Hour has non-monotonic, cyclical, context-dependent patterns (bimodal on workdays, unimodal on weekends). Solutions: one-hot encoding, sin/cos encoding, rush hour features.

---
## Generate Submission File

In [23]:
X_test = test_fe.reindex(columns=feature_cols_eng, fill_value=0)
X_test_scaled = scaler_eng.transform(X_test)
y_test_pred = np.clip(np.expm1(best_final_model.predict(X_test_scaled)), 0, None).round().astype(int)
submission = pd.DataFrame({'datetime': test_fe['datetime'].dt.strftime('%d-%m-%Y %-H:%M'), 'count_predicted': y_test_pred})
submission.to_csv('submission.csv', index=False)
print(f"Submission: {len(submission)} rows, min={y_test_pred.min()}, max={y_test_pred.max()}, mean={y_test_pred.mean():.1f}")

Submission: 2613 rows, min=1, max=910, mean=183.4


---
## Summary

This assignment explored Bike Sharing Demand Prediction using progressively sophisticated regression techniques. Simple LR served as baseline. Feature engineering (hour dummies, rush hour indicators, cyclical encoding) dramatically improved performance. Log-transforming the target aligned with RMSLE. Ridge regularization controlled overfitting. The final model (Ridge + Log + Engineered Features) achieved the best RMSLE of 0.5485.